# Steam Top Games 2026 Research

This notebook explores the Steam Top Games 2026 dataset and trains machine learning models to predict `peak_ccu`.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
data = pd.read_csv('../data/steam_top_games_2026.csv')
data.head()

,app_id,name,release_date,coming_soon,price_usd,is_free,discount_pct,developer,publisher,genres,...,estimated_owners,avg_playtime_forever,avg_playtime_2weeks,median_playtime,peak_ccu,required_age,dlc_count,achievements,short_description,header_image
0,794260,Outward Definitive Edition,"May 17, 2022",False,4.79,False,88,Nine Dots Studio,"Prime Matter, Deep Silver",RPG,...,"1,000,000 .. 2,000,000",1332,824,465,469,0,1,72,No remarkable journey is achieved without grea...,https://shared.akamai.steamstatic.com/store_it...
1,253920,Gorky 17,"Sep 27, 2013",False,9.99,False,0,Metropolis Software,TopWare Interactive,"RPG, Strategy",...,"200,000 .. 500,000",301,0,328,61,17,2,0,November 2008. NATO intelligence services repo...,https://shared.akamai.steamstatic.com/store_it...
2,613010,Secret in Story,"Jun 19, 2017",False,0.89,False,10,Naivus Luo,Naivus Luo,"Adventure, Indie",...,"2,000,000 .. 5,000,000",251,0,243,0,0,0,32,"Accompanied by beautiful piano music, you begi...",https://shared.akamai.steamstatic.com/store_it...
3,892420,懒人修仙传,"Nov 14, 2018",False,3.99,False,0,托更的修罗,托更的修罗,"Casual, Indie, RPG, Simulation",...,"200,000 .. 500,000",5786,0,9223,41,0,0,0,这是一款很&quot;休闲&quot;的文字挂机游戏，游戏小而系统完善，玩法丰富，极其耗电，...,https://shared.akamai.steamstatic.com/store_it...
4,914010,Train Station Renovation,"Oct 1, 2020",False,18.99,False,0,Live Motion Games,"Live Motion Games, Frozen Way, PlayWay S.A., F...","Casual, Indie, Simulation",...,"200,000 .. 500,000",448,0,201,16,0,1,73,"Welcome to an old, ruined train station. A pla...",https://shared.akamai.steamstatic.com/store_it...


In [8]:
print(data.shape)
data.isnull().sum().sort_values(ascending=False).head(10)

(1495, 29)


metacritic_score     955
publisher             12
categories             9
developer              8
genres                 8
release_date           7
tags                   6
estimated_owners       5
short_description      2
header_image           1
dtype: int64

## Feature Engineering

We create new useful features from dates, owner ranges, and review columns.

In [9]:
def parse_owners_midpoint(value):
    if pd.isna(value):
        return np.nan
    numbers = re.findall(r'[\d,]+', str(value))
    if len(numbers) >= 2:
        low = int(numbers[0].replace(',', ''))
        high = int(numbers[1].replace(',', ''))
        return (low + high) / 2
    return np.nan

data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
data['release_year'] = data['release_date'].dt.year
data['game_age_years'] = 2026 - data['release_year']
data['owners_midpoint'] = data['estimated_owners'].apply(parse_owners_midpoint)
data['total_reviews'] = data['positive_reviews'] + data['negative_reviews']
data['positive_review_ratio'] = np.where(data['total_reviews'] > 0, data['positive_reviews'] / data['total_reviews'], np.nan)
data['main_genre'] = data['genres'].fillna('Unknown').apply(lambda value: str(value).split(',')[0].strip())
data[['release_year', 'game_age_years', 'owners_midpoint', 'total_reviews', 'positive_review_ratio', 'main_genre']].head()

,release_year,game_age_years,owners_midpoint,total_reviews,positive_review_ratio,main_genre
0,2022.0,4.0,1500000.0,28455,0.728800,RPG
1,NaN,NaN,350000.0,2027,0.797237,RPG
2,NaN,NaN,3500000.0,114,0.692982,Adventure
3,NaN,NaN,350000.0,1278,0.601721,Casual
4,NaN,NaN,350000.0,2620,0.822137,Casual


## Model Training

The target variable is `peak_ccu`, so this is a regression task.

In [16]:
target = 'peak_ccu'
features = ['price_usd', 'is_free', 'discount_pct', 'metacritic_score', 'recommendations', 'positive_reviews', 'negative_reviews', 'avg_playtime_forever', 'avg_playtime_2weeks', 'median_playtime', 'required_age', 'dlc_count', 'achievements', 'platforms_win', 'platforms_mac', 'platforms_linux', 'release_year', 'game_age_years', 'owners_midpoint', 'total_reviews', 'positive_review_ratio', 'main_genre']

X = data[features].copy()
# Convert bool columns to int for preprocessing
X['is_free'] = X['is_free'].astype(int)
X['platforms_win'] = X['platforms_win'].astype(int)
X['platforms_mac'] = X['platforms_mac'].astype(int)
X['platforms_linux'] = X['platforms_linux'].astype(int)

y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), numerical_cols),
    ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
])

In [17]:
print("Numerical columns:", numerical_cols)
print("\nCategorical columns:", categorical_cols)
print("\nData types in X:")
print(X.dtypes)
print("\nMissing values in categorical columns:")
print(X[categorical_cols].isnull().sum())
print("\nMissing values in numerical columns:")
print(X[numerical_cols].isnull().sum())

Numerical columns: ['price_usd', 'is_free', 'discount_pct', 'metacritic_score', 'recommendations', 'positive_reviews', 'negative_reviews', 'avg_playtime_forever', 'avg_playtime_2weeks', 'median_playtime', 'required_age', 'dlc_count', 'achievements', 'platforms_win', 'platforms_mac', 'platforms_linux', 'release_year', 'game_age_years', 'owners_midpoint', 'total_reviews', 'positive_review_ratio']

Categorical columns: ['main_genre']

Data types in X:
price_usd                float64
is_free                    int64
discount_pct               int64
metacritic_score         float64
recommendations            int64
positive_reviews           int64
negative_reviews           int64
avg_playtime_forever       int64
avg_playtime_2weeks        int64
median_playtime            int64
required_age               int64
dlc_count                  int64
achievements               int64
platforms_win              int64
platforms_mac              int64
platforms_linux            int64
release_year       

In [18]:
def evaluate_model(name, model):
    pipeline = Pipeline(steps=[('preprocess', preprocessor), ('model', model)])
    regressor = TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)
    regressor.fit(X_train, y_train)
    predictions = np.maximum(regressor.predict(X_test), 0)
    return {
        'model': name,
        'MAE': round(mean_absolute_error(y_test, predictions), 2),
        'RMSE': round(mean_squared_error(y_test, predictions) ** 0.5, 2),
        'R2': round(r2_score(y_test, predictions), 3)
    }, regressor

models = [
    ('Baseline Median Model', DummyRegressor(strategy='median')),
    ('Decision Tree Regressor', DecisionTreeRegressor(random_state=42, max_depth=6)),
    ('Random Forest Regressor', RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=1))
]

results = []
trained_models = {}

for name, model in models:
    metrics, trained = evaluate_model(name, model)
    results.append(metrics)
    trained_models[name] = trained

pd.DataFrame(results)

,model,MAE,RMSE,R2
0,Baseline Median Model,910.05,8517.95,-0.011
1,Decision Tree Regressor,852.91,7689.66,0.176
2,Random Forest Regressor,656.51,5886.55,0.517
